# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant dataset schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
ds = mlc.Dataset(croissant_url)

# Print dataset name and description
print(f"{ds.metadata.name}: {ds.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Show the record sets defined in the dataset (by their @id)
print("Available record sets (@id):")
for record_set in ds.metadata.record_sets:
    print(f"- {record_set['@id']} : {record_set.get('name', record_set.get('description',''))}")

# For each record set, display its fields (by their @id and name)
for record_set in ds.metadata.record_sets:
    print(f"\nRecord Set: {record_set['@id']}")
    print("Fields:")
    if 'fields' in record_set:
        for field in record_set['fields']:
            desc = field.get('name', field.get('description',''))
            print(f"    - {field['@id']}: {desc}")
    else:
        print("    (No field information in this record set)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Let's build a list of record set @ids
record_set_ids = [rs['@id'] for rs in ds.metadata.record_sets]
print("Record sets found:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    try:
        # Use records iterator for each record set
        records = list(ds.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} rows for record set '{record_set_id}'")
        else:
            print(f"Record set '{record_set_id}' contains no records.")
    except Exception as e:
        print(f"Error loading '{record_set_id}': {e}")

# Pick the first record set as the one we'll demo for EDA
if record_set_ids and record_set_ids[0] in dataframes:
    primary_record_set = record_set_ids[0]
else:
    primary_record_set = None

if primary_record_set:
    print('Columns in primary record set:', dataframes[primary_record_set].columns.tolist())
    display(dataframes[primary_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

> **Note:** All fields/columns are referenced by their `@id`. Use the overview cells above to pick actual field `@id`s.

In [ ]:
# EDA Example: Filter by a numeric field, normalize it, and group by a categorical field

if primary_record_set:
    df = dataframes[primary_record_set].copy()

    # Try to auto-select a likely numeric field
    # We'll filter columns to those that look numeric by the data type in the first row
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if not numeric_candidates:
        # Try to coerce all columns to numeric, keep those that succeed
        for col in df.columns:
            try:
                df[col+'_numeric'] = pd.to_numeric(df[col], errors='coerce')
            except Exception:
                continue
        numeric_candidates = [col for col in df.columns if col.endswith('_numeric')]

    print("Numeric field candidates (@id):", numeric_candidates)

    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a likely categorical/text field
        group_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
        print("Grouping candidates (@id):", group_candidates)
        if group_candidates:
            group_field = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
    else:
        print("No numeric field candidates found for EDA.")
else:
    print("No primary record set loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_record_set and numeric_candidates:
    # Plot the distribution of the selected numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_candidates:
        # Boxplot by the first group candidate
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_candidates[0]], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_candidates[0]}")
        plt.xlabel(group_candidates[0])
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we've demonstrated loading the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using `mlcroissant`, extracting its tables (referencing all entities by `@id`), and performing basic exploratory data analysis on its main record set. To conduct a deeper domain-specific analysis, review the dataset documentation and field definitions at the source, and adapt this notebook accordingly for your research question.*